<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


Estimated time needed: **45** minutes


In this lab, you will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


After completing this lab, you will be able to:


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [1]:
!pip install pandas
!pip install matplotlib

## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [2]:
# Import necessary libraries
import pandas as pd

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [5]:
# Write your code here
summary = pd.DataFrame({
    'Data type': df.dtypes.astype(str),
    'Non-null count': df.notna().sum(),
    'Missing count': df.isna().sum(),
    'Missing percentage': (df.isna().mean() * 100).round(2)
})
print(summary)


                    Data type  Non-null count  Missing count  \
ResponseId              int64           65437              0   
MainBranch                str           65437              0   
Age                       str           65437              0   
Employment                str           65437              0   
RemoteWork                str           54806          10631   
...                       ...             ...            ...   
JobSatPoints_11       float64           29445          35992   
SurveyLength              str           56182           9255   
SurveyEase                str           56238           9199   
ConvertedCompYearly   float64           23435          42002   
JobSat                float64           29126          36311   

                     Missing percentage  
ResponseId                         0.00  
MainBranch                         0.00  
Age                                0.00  
Employment                         0.00  
RemoteWork           

<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [6]:
# Write your code here
print(df.describe())

         ResponseId      CompTotal       WorkExp  JobSatPoints_1  \
count  65437.000000   3.374000e+04  29658.000000    29324.000000   
mean   32719.000000  2.963841e+145     11.466957       18.581094   
std    18890.179119  5.444117e+147      9.168709       25.966221   
min        1.000000   0.000000e+00      0.000000        0.000000   
25%    16360.000000   6.000000e+04      4.000000        0.000000   
50%    32719.000000   1.100000e+05      9.000000       10.000000   
75%    49078.000000   2.500000e+05     16.000000       22.000000   
max    65437.000000  1.000000e+150     50.000000      100.000000   

       JobSatPoints_4  JobSatPoints_5  JobSatPoints_6  JobSatPoints_7  \
count    29393.000000    29411.000000    29450.000000     29448.00000   
mean         7.522140       10.060857       24.343232        22.96522   
std         18.422661       21.833836       27.089360        27.01774   
min          0.000000        0.000000        0.000000         0.00000   
25%          0.000000 

### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [8]:
# Write your code here
# Inspect the most frequent Country values and possible missing values
country_counts=df['Country'].value_counts(dropna=False)
print(country_counts.head(20))
print('Missing Country values', df['Country'].isna().sum())

# Entries with very low frequency are displayed for review rather than
# removed automatically, because rare countries may be valid responses.
print('Countries with at most 2 responses:')
print(country_counts[country_counts <= 2])

Country
United States of America                                11095
NaN                                                      6507
Germany                                                  4947
India                                                    4231
United Kingdom of Great Britain and Northern Ireland     3224
Ukraine                                                  2672
France                                                   2110
Canada                                                   2104
Poland                                                   1534
Netherlands                                              1449
Brazil                                                   1375
Italy                                                    1341
Australia                                                1260
Spain                                                    1123
Sweden                                                   1016
Russian Federation                                        925


<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [9]:
## Write your code here
# Standardize clearly equivalent text values without removing valid records
country_standardization = {
    'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'Türkiye': 'Turkey'
}
df['Country']=df['Country'].replace(country_standardization)
df['Country']=df['Country'].str.strip()

df['EdLevel']=df['EdLevel'].str.strip()

print(df[['Country','EdLevel']].head())

          Country                                            EdLevel
0   United States                          Primary/elementary school
1  United Kingdom       Bachelor’s degree (B.A., B.S., B.Eng., etc.)
2  United Kingdom    Master’s degree (M.A., M.S., M.Eng., MBA, etc.)
3          Canada  Some college/university study without earning ...
4          Norway  Secondary school (e.g. American high school, G...


### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [10]:
## Write your code here
employment_dummies = pd.get_dummies(
    df['Employment'],
    prefix='Employment',
    dummy_na=True,
    dtype=int
)
df=pd.concat([df,employment_dummies],axis=1)
print(employment_dummies.head())

   Employment_Employed, full-time  \
0                               1   
1                               1   
2                               1   
3                               0   
4                               0   

   Employment_Employed, full-time;Employed, part-time  \
0                                                  0    
1                                                  0    
2                                                  0    
3                                                  0    
4                                                  0    

   Employment_Employed, full-time;Independent contractor, freelancer, or self-employed  \
0                                                  0                                     
1                                                  0                                     
2                                                  0                                     
3                                                  0                      

### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [12]:
## Write your code here
# Identify columns with the largest number of missing values
missing_summary=(
    df.isna().sum().sort_values(ascending=False).rename('MissingCount')
)
missing_summary=missing_summary[missing_summary>0]
print(missing_summary.head())

AINextMuch less integrated    64289
AINextLess integrated         63082
AINextNo change               52939
AINextMuch more integrated    51999
EmbeddedAdmired               48704
Name: MissingCount, dtype: int64


<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [13]:
## Write your code here
# Impute the numeric compensation column with its median
df['ConvertedCompYearly'] = pd.to_numeric(
    df['ConvertedCompYearly'], errors='coerce'
)
compensation_median=df['ConvertedCompYearly'].median()
df['ConvertedCompYearly']=df['ConvertedCompYearly'].fillna(compensation_median)
print('Compensation median used:',compensation_median)
print('Missing ConvertedCompYearly after imputation:', df['ConvertedCompYearly'].isna().sum())

Compensation median used: 65000.0
Missing ConvertedCompYearly after imputation: 0


<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [14]:
## Write your code here
# Impute missing RemoteWork values with the most frequent category
remote_work_mode = df['RemoteWork'].mode(dropna=True) # Get the most frequent value, ignoring NaN
if remote_work_mode.empty:
    raise ValueError('RemoteWork has no non-missing value for imputation.') # Handle the case where there are no non-missing values
remote_work_fill_value = remote_work_mode.iloc[0] # Get the first (and only) value from the mode result
df['RemoteWork'] = df['RemoteWork'].fillna(remote_work_fill_value) # Impute missing RemoteWork values with the most frequent category
print('RemoteWork value used:', remote_work_fill_value)
print('Missing RemoteWork after imputation:',
      df['RemoteWork'].isna().sum())


RemoteWork value used: Hybrid (some remote, some in-person)
Missing RemoteWork after imputation: 0


### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [15]:
## Write your code here
# Min-Max scaling of the imputed compensation column
comp_min = df['ConvertedCompYearly'].min()
comp_max = df['ConvertedCompYearly'].max()
if comp_max == comp_min:
    df['ConvertedCompYearly_MinMax'] = 0.0
else:
    df['ConvertedCompYearly_MinMax'] = (
        df['ConvertedCompYearly'] - comp_min
    ) / (comp_max - comp_min)
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_MinMax']].head())

   ConvertedCompYearly  ConvertedCompYearly_MinMax
0              65000.0                    0.003998
1              65000.0                    0.003998
2              65000.0                    0.003998
3              65000.0                    0.003998
4              65000.0                    0.003998


<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [17]:
## Write your code here
import numpy as np
# Log transformation using log1p, which is safe for zero values
df['ConvertedCompYearly_Log'] = np.log1p(df['ConvertedCompYearly']) # Use log1p to handle zero values safely
print(df[['ConvertedCompYearly', 'ConvertedCompYearly_Log']].head()) # Display the first few rows of the log-transformed data

   ConvertedCompYearly  ConvertedCompYearly_Log
0              65000.0                11.082158
1              65000.0                11.082158
2              65000.0                11.082158
3              65000.0                11.082158
4              65000.0                11.082158


### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [19]:
## Write your code here
# Convert YearsCodePro to numeric while preserving the original column
years_code_pro_numeric = (
    df['YearsCodePro']
      .replace({'Less than 1 year': '0.5', 'More than 50 years': '51'}) # Replace special cases with numeric values
      .pipe(pd.to_numeric, errors='coerce') # Convert to numeric, coercing errors to NaN
)

def classify_experience(years): 
    if pd.isna(years):
        return 'Unknown'
    if years < 3:
        return 'Beginner'
    if years < 8:
        return 'Intermediate'
    return 'Experienced'

df['ExperienceLevel'] = years_code_pro_numeric.map(classify_experience) # Map the numeric values to experience levels
print(df[['YearsCodePro', 'ExperienceLevel']].head()) # Display the first few rows of the experience level data
print(df['ExperienceLevel'].value_counts(dropna=False)) # Display the count of each experience level

  YearsCodePro ExperienceLevel
0          NaN         Unknown
1           17     Experienced
2           27     Experienced
3          NaN         Unknown
4          NaN         Unknown
ExperienceLevel
Experienced     25753
Intermediate    16194
Unknown         13827
Beginner         9663
Name: count, dtype: int64


### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
